# Часть 1. Обработка данных и разведочный анализ

В этом блоке:
- загружаем и валидируем сырые данные;
- очищаем пропуски, типы и выбросы;
- строим базовые аналитические витрины;
- рассчитываем RFM, ABC/XYZ, логистические и поведенческие признаки.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')
sns.set_theme(style='whitegrid', palette='Set2')

In [ ]:
# Укажите пути к файлам в Kaggle.
# Примеры:
# /kaggle/input/your-dataset/orders_enriched.csv
# /kaggle/input/your-dataset/events.csv

ORDERS_PATH = '/kaggle/input/your-dataset/orders_enriched.csv'
EVENTS_PATH = '/kaggle/input/your-dataset/events.csv'

DATE_COLS_ORDERS = ['created_at', 'returned_at', 'shipped_at', 'delivered_at', 'sold_at']
DATE_COLS_EVENTS = ['created_at']

In [ ]:
orders = pd.read_csv(ORDERS_PATH)
events = pd.read_csv(EVENTS_PATH)

print('orders shape:', orders.shape)
print('events shape:', events.shape)

display(orders.head(3))
display(events.head(3))

In [ ]:
def basic_overview(df, name='df'):
    print(f'===== {name} =====')
    print('shape:', df.shape)
    display(df.dtypes.to_frame('dtype').T)
    display(df.isna().mean().sort_values(ascending=False).mul(100).round(2).to_frame('missing_pct').head(20))
    print('duplicates:', df.duplicated().sum())
    print()

def missing_report(df):
    report = pd.DataFrame({
        'missing_cnt': df.isna().sum(),
        'missing_pct': df.isna().mean().mul(100)
    }).sort_values('missing_pct', ascending=False)
    return report

def describe_num(df):
    num_cols = df.select_dtypes(include=['number', 'bool']).columns
    return df[num_cols].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T

basic_overview(orders, 'orders_raw')
basic_overview(events, 'events_raw')

## 1. Приведение типов и первичная очистка

In [ ]:
orders = orders.copy()
events = events.copy()

for col in DATE_COLS_ORDERS:
    if col in orders.columns:
        orders[col] = pd.to_datetime(orders[col], errors='coerce', utc=True)

for col in DATE_COLS_EVENTS:
    if col in events.columns:
        events[col] = pd.to_datetime(events[col], errors='coerce', utc=True)

for col in orders.select_dtypes(include='object').columns:
    orders[col] = orders[col].astype(str).str.strip()
    orders.loc[orders[col].isin(['nan', 'None', '']), col] = np.nan

for col in events.select_dtypes(include='object').columns:
    events[col] = events[col].astype(str).str.strip()
    events.loc[events[col].isin(['nan', 'None', '']), col] = np.nan

if 'user_id' in events.columns:
    events['user_id'] = pd.to_numeric(events['user_id'], errors='coerce').astype('Int64')

if 'age' in orders.columns:
    orders['age'] = pd.to_numeric(orders['age'], errors='coerce')

numeric_candidate_cols = [
    'sale_price', 'cost', 'retail_price', 'product_retail_price',
    'delivery_longitude', 'delivery_latitude', 'warehouse_longitude', 'warehouse_latitude',
    'num_of_item'
]
for col in numeric_candidate_cols:
    if col in orders.columns:
        orders[col] = pd.to_numeric(orders[col], errors='coerce')

print('types converted')

In [ ]:
display(missing_report(orders).head(25))
display(missing_report(events).head(20))

In [ ]:
# Удаляем полные дубликаты и технически бесполезные поля.
orders = orders.drop_duplicates().copy()
events = events.drop_duplicates().copy()

if 'sold_at' in orders.columns and orders['sold_at'].isna().all():
    orders = orders.drop(columns=['sold_at'])

# Заполняем пропуски в категориальных полях безопасными значениями.
fill_unknown_cols = [
    'city', 'brand', 'product_brand', 'product_name',
    'product_name_clean', 'customer_review', 'traffic_source', 'state'
]
for col in fill_unknown_cols:
    if col in orders.columns:
        orders[col] = orders[col].fillna('Unknown')

event_fill_unknown = ['city', 'state', 'browser', 'traffic_source', 'event_type', 'uri']
for col in event_fill_unknown:
    if col in events.columns:
        events[col] = events[col].fillna('Unknown')

# Если очищенное имя товара пропущено, подставляем исходное.
if {'product_name_clean', 'product_name'}.issubset(orders.columns):
    orders['product_name_clean'] = orders['product_name_clean'].fillna(orders['product_name'])

# Возраст и количество товаров ограничиваем разумными границами.
if 'age' in orders.columns:
    orders.loc[(orders['age'] < 14) | (orders['age'] > 100), 'age'] = np.nan
    orders['age'] = orders['age'].fillna(orders['age'].median())

if 'num_of_item' in orders.columns:
    orders.loc[orders['num_of_item'] <= 0, 'num_of_item'] = 1

print('duplicates removed and basic imputations done')

## 2. Выбросы и контроль качества числовых признаков

In [ ]:
display(describe_num(orders).head(20))

In [ ]:
def winsorize_series(s, q_low=0.01, q_high=0.99):
    low = s.quantile(q_low)
    high = s.quantile(q_high)
    return s.clip(lower=low, upper=high)

winsor_cols = ['sale_price', 'cost', 'retail_price', 'product_retail_price']
for col in winsor_cols:
    if col in orders.columns:
        orders[col] = winsorize_series(orders[col].astype(float))

# Убираем невозможные отрицательные значения.
for col in ['sale_price', 'cost', 'retail_price', 'product_retail_price']:
    if col in orders.columns:
        orders.loc[orders[col] < 0, col] = np.nan
        orders[col] = orders[col].fillna(orders[col].median())

print('outliers clipped')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
plot_cols = [c for c in ['sale_price', 'cost', 'retail_price', 'age'] if c in orders.columns]
axes = axes.flatten()

for i, col in enumerate(plot_cols):
    sns.boxplot(x=orders[col], ax=axes[i])
    axes[i].set_title(col)

for j in range(len(plot_cols), len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

## 3. Базовые производные признаки по транзакциям и логистике

In [ ]:
orders['order_date'] = orders['created_at'].dt.date
orders['order_month'] = orders['created_at'].dt.to_period('M').astype(str)
orders['order_week'] = orders['created_at'].dt.isocalendar().week.astype('int64')
orders['day_of_week'] = orders['created_at'].dt.day_name()
orders['hour'] = orders['created_at'].dt.hour

orders['line_revenue'] = orders['sale_price'] * orders['num_of_item']
orders['line_cost'] = orders['cost'] * orders['num_of_item']
orders['line_profit'] = orders['line_revenue'] - orders['line_cost']
orders['discount_ratio'] = np.where(
    orders['retail_price'] > 0,
    1 - orders['sale_price'] / orders['retail_price'],
    0
)

orders['is_returned'] = orders['returned_at'].notna().astype(int)
orders['is_shipped'] = orders['shipped_at'].notna().astype(int)
orders['is_delivered'] = orders['delivered_at'].notna().astype(int)
orders['is_cancelled'] = orders['status'].astype(str).str.lower().str.contains('cancel').fillna(False).astype(int)

orders['shipping_delay_days'] = (orders['shipped_at'] - orders['created_at']).dt.total_seconds() / 86400
orders['delivery_days'] = (orders['delivered_at'] - orders['created_at']).dt.total_seconds() / 86400
orders['return_days'] = (orders['returned_at'] - orders['created_at']).dt.total_seconds() / 86400

for col in ['shipping_delay_days', 'delivery_days', 'return_days']:
    orders.loc[orders[col] < 0, col] = np.nan

orders['delivery_gap_days'] = (orders['delivered_at'] - orders['shipped_at']).dt.total_seconds() / 86400
orders.loc[orders['delivery_gap_days'] < 0, 'delivery_gap_days'] = np.nan

orders[['shipping_delay_days', 'delivery_days', 'return_days', 'delivery_gap_days']].describe().T

## 4. Первичный EDA

In [ ]:
daily_sales = orders.groupby('order_date', as_index=False).agg(
    revenue=('line_revenue', 'sum'),
    orders_cnt=('order_id', 'nunique'),
    users_cnt=('user_id', 'nunique')
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.lineplot(data=daily_sales, x='order_date', y='revenue', ax=axes[0])
axes[0].set_title('Динамика выручки')
axes[0].tick_params(axis='x', rotation=45)

sns.lineplot(data=daily_sales, x='order_date', y='orders_cnt', ax=axes[1])
axes[1].set_title('Динамика количества заказов')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
top_categories = orders.groupby('product_category', as_index=False).agg(
    revenue=('line_revenue', 'sum'),
    orders_cnt=('order_id', 'nunique'),
    return_rate=('is_returned', 'mean')
).sort_values('revenue', ascending=False).head(15)

display(top_categories)

plt.figure(figsize=(12, 6))
sns.barplot(data=top_categories, y='product_category', x='revenue')
plt.title('Топ категорий по выручке')
plt.show()

In [ ]:
geo_sales = orders.groupby(['state', 'city'], as_index=False).agg(
    revenue=('line_revenue', 'sum'),
    users_cnt=('user_id', 'nunique'),
    avg_delivery_days=('delivery_days', 'mean')
).sort_values('revenue', ascending=False).head(20)

display(geo_sales.head(10))

## 5. Качество товаров и признаки недовольства

In [ ]:
negative_words = [
    'bad', 'poor', 'terrible', 'awful', 'disappointed', 'broken', 'late', 'return',
    'defect', 'cheap', 'worst', 'hate', 'problem', 'issue', 'damaged'
]

review_text = orders['customer_review'].fillna('').str.lower()
orders['review_negative_flag'] = review_text.apply(lambda x: int(any(word in x for word in negative_words)))

product_quality = orders.groupby(['product_id', 'product_name_clean', 'product_category'], as_index=False).agg(
    revenue=('line_revenue', 'sum'),
    orders_cnt=('order_id', 'nunique'),
    return_rate=('is_returned', 'mean'),
    negative_review_rate=('review_negative_flag', 'mean'),
    avg_delivery_days=('delivery_days', 'mean')
)

product_quality['quality_risk_score'] = (
    0.5 * product_quality['return_rate'].fillna(0) +
    0.3 * product_quality['negative_review_rate'].fillna(0) +
    0.2 * (product_quality['avg_delivery_days'].fillna(product_quality['avg_delivery_days'].median()) / product_quality['avg_delivery_days'].fillna(product_quality['avg_delivery_days'].median()).max())
)

display(product_quality.sort_values('quality_risk_score', ascending=False).head(20))

## 6. RFM-сегментация

In [ ]:
snapshot_date = orders['created_at'].max() + pd.Timedelta(days=1)

rfm = orders.groupby('user_id', as_index=False).agg(
    last_purchase=('created_at', 'max'),
    frequency=('order_id', 'nunique'),
    monetary=('line_revenue', 'sum')
)

rfm['recency'] = (snapshot_date - rfm['last_purchase']).dt.days
rfm = rfm[['user_id', 'recency', 'frequency', 'monetary']].copy()

rfm['r_score'] = pd.qcut(rfm['recency'].rank(method='first', ascending=False), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['f_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['m_score'] = pd.qcut(rfm['monetary'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['rfm_score'] = rfm[['r_score', 'f_score', 'm_score']].astype(str).agg(''.join, axis=1)

def rfm_segment(row):
    if row['r_score'] >= 4 and row['f_score'] >= 4 and row['m_score'] >= 4:
        return 'Champions'
    if row['r_score'] >= 3 and row['f_score'] >= 3:
        return 'Loyal'
    if row['r_score'] >= 4 and row['f_score'] <= 2:
        return 'Potential Loyalist'
    if row['r_score'] <= 2 and row['f_score'] >= 3:
        return 'At Risk'
    if row['r_score'] <= 2 and row['f_score'] <= 2:
        return 'Hibernating'
    return 'Regular'

rfm['rfm_segment'] = rfm.apply(rfm_segment, axis=1)
display(rfm.head())
display(rfm['rfm_segment'].value_counts(dropna=False).to_frame('users_cnt'))

## 7. ABC/XYZ-анализ по товарам

In [ ]:
monthly_product_sales = orders.groupby(['product_id', 'product_name_clean', 'order_month'], as_index=False).agg(
    revenue=('line_revenue', 'sum')
)

abc = monthly_product_sales.groupby(['product_id', 'product_name_clean'], as_index=False).agg(
    total_revenue=('revenue', 'sum'),
    revenue_std=('revenue', 'std'),
    revenue_mean=('revenue', 'mean')
)

abc['revenue_std'] = abc['revenue_std'].fillna(0)
abc['revenue_mean'] = abc['revenue_mean'].replace(0, np.nan)
abc['cv'] = (abc['revenue_std'] / abc['revenue_mean']).fillna(0)

abc = abc.sort_values('total_revenue', ascending=False)
abc['cum_share'] = abc['total_revenue'].cumsum() / abc['total_revenue'].sum()

abc['abc_class'] = np.select(
    [abc['cum_share'] <= 0.80, abc['cum_share'] <= 0.95],
    ['A', 'B'],
    default='C'
)

abc['xyz_class'] = np.select(
    [abc['cv'] <= 0.10, abc['cv'] <= 0.25],
    ['X', 'Y'],
    default='Z'
)

abc['abc_xyz'] = abc['abc_class'] + abc['xyz_class']
display(abc.head(20))
display(abc['abc_xyz'].value_counts().to_frame('products_cnt'))

## 8. Поведенческие признаки из событий

In [ ]:
events = events.dropna(subset=['user_id']).copy()
events['user_id'] = events['user_id'].astype('int64')
events['event_date'] = events['created_at'].dt.date
events['event_month'] = events['created_at'].dt.to_period('M').astype(str)

event_user_features = events.groupby('user_id', as_index=False).agg(
    events_cnt=('id', 'count'),
    unique_sessions=('session_id', 'nunique'),
    active_days=('event_date', 'nunique'),
    last_event_at=('created_at', 'max'),
    favorite_browser=('browser', lambda x: x.mode().iloc[0] if not x.mode().empty else 'Unknown'),
    favorite_event_type=('event_type', lambda x: x.mode().iloc[0] if not x.mode().empty else 'Unknown')
)

event_type_pivot = pd.crosstab(events['user_id'], events['event_type'])
event_type_pivot.columns = [f'event_type_{str(col).lower().replace(" ", "_")}_cnt' for col in event_type_pivot.columns]
event_type_pivot = event_type_pivot.reset_index()

event_user_features = event_user_features.merge(event_type_pivot, on='user_id', how='left')
event_user_features['days_since_last_event'] = (events['created_at'].max() - event_user_features['last_event_at']).dt.days

display(event_user_features.head())

## 9. Клиентская витрина признаков для churn и recommendation

In [ ]:
user_features = orders.groupby('user_id', as_index=False).agg(
    first_order_at=('created_at', 'min'),
    last_order_at=('created_at', 'max'),
    orders_cnt=('order_id', 'nunique'),
    items_cnt=('num_of_item', 'sum'),
    total_revenue=('line_revenue', 'sum'),
    total_profit=('line_profit', 'sum'),
    avg_order_value=('line_revenue', 'mean'),
    avg_items_per_line=('num_of_item', 'mean'),
    return_rate=('is_returned', 'mean'),
    cancel_rate=('is_cancelled', 'mean'),
    avg_delivery_days=('delivery_days', 'mean'),
    avg_shipping_delay_days=('shipping_delay_days', 'mean'),
    discount_ratio_mean=('discount_ratio', 'mean'),
    unique_categories=('product_category', 'nunique'),
    unique_brands=('product_brand', 'nunique'),
    favorite_category=('product_category', lambda x: x.mode().iloc[0] if not x.mode().empty else 'Unknown'),
    favorite_brand=('product_brand', lambda x: x.mode().iloc[0] if not x.mode().empty else 'Unknown'),
    age=('age', 'median'),
    gender=('gender', lambda x: x.mode().iloc[0] if not x.mode().empty else 'Unknown'),
    state=('state', lambda x: x.mode().iloc[0] if not x.mode().empty else 'Unknown'),
    city=('city', lambda x: x.mode().iloc[0] if not x.mode().empty else 'Unknown'),
    traffic_source=('traffic_source', lambda x: x.mode().iloc[0] if not x.mode().empty else 'Unknown'),
    is_loyal=('is_loyal', 'max')
)

user_features['customer_tenure_days'] = (snapshot_date - user_features['first_order_at']).dt.days
user_features['days_since_last_order'] = (snapshot_date - user_features['last_order_at']).dt.days
user_features['purchase_frequency_per_30d'] = user_features['orders_cnt'] / np.maximum(user_features['customer_tenure_days'] / 30, 1)

category_share = orders.groupby(['user_id', 'product_category']).size().rename('cnt').reset_index()
category_share['rank'] = category_share.groupby('user_id')['cnt'].rank(method='first', ascending=False)
top_cat = category_share[category_share['rank'] == 1][['user_id', 'cnt']].rename(columns={'cnt': 'top_category_purchases'})
total_cat = category_share.groupby('user_id', as_index=False)['cnt'].sum().rename(columns={'cnt': 'total_category_purchases'})
cat_concentration = top_cat.merge(total_cat, on='user_id', how='left')
cat_concentration['favorite_category_share'] = cat_concentration['top_category_purchases'] / cat_concentration['total_category_purchases']

user_features = user_features.merge(cat_concentration[['user_id', 'favorite_category_share']], on='user_id', how='left')
user_features = user_features.merge(rfm[['user_id', 'recency', 'frequency', 'monetary', 'rfm_score', 'rfm_segment']], on='user_id', how='left')
user_features = user_features.merge(event_user_features, on='user_id', how='left')

display(user_features.head())
print('user_features shape:', user_features.shape)

## 10. Базовая сегментация клиентов

In [ ]:
cluster_cols = [
    'recency', 'frequency', 'monetary', 'return_rate', 'avg_delivery_days',
    'discount_ratio_mean', 'unique_categories', 'favorite_category_share',
    'events_cnt', 'unique_sessions', 'days_since_last_event'
]
cluster_cols = [c for c in cluster_cols if c in user_features.columns]

cluster_df = user_features[cluster_cols].copy()
cluster_df = cluster_df.fillna(cluster_df.median(numeric_only=True))

scaler = StandardScaler()
cluster_scaled = scaler.fit_transform(cluster_df)

kmeans = KMeans(n_clusters=5, random_state=42, n_init=20)
user_features['cluster'] = kmeans.fit_predict(cluster_scaled)

cluster_summary = user_features.groupby('cluster')[cluster_cols].mean().round(2)
display(cluster_summary)
display(user_features['cluster'].value_counts().sort_index().to_frame('users_cnt'))

## 11. Финальные таблицы для следующих этапов

In [ ]:
sales_mart = daily_sales.copy()
product_mart = product_quality.merge(abc[['product_id', 'abc_class', 'xyz_class', 'abc_xyz']], on='product_id', how='left')
customer_mart = user_features.copy()

print('sales_mart:', sales_mart.shape)
print('product_mart:', product_mart.shape)
print('customer_mart:', customer_mart.shape)

display(customer_mart.head(3))
display(product_mart.head(3))

## 12. Что важно зафиксировать в выводах по первой задаче

После выполнения ноутбука опишите:
- структуру данных и ключевые проблемы качества;
- долю пропусков и что именно было заполнено, удалено или ограничено;
- топ категорий/регионов/источников трафика по выручке и активности;
- какие группы клиентов видны через RFM и кластеризацию;
- какие товары и категории имеют высокий риск недовольства;
- какие признаки будут использованы дальше в churn и recommendation.